# UACB-604 · Lecture 2.1 — Simulating Sanger sequencing

In Sanger sequencing a polymerase copies the template until, by chance, it adds a **dideoxy** nucleotide (ddNTP) that stops the chain.
Each fragment ends with a coloured ddNTP; sorting fragments by length (capillary electrophoresis) spells out the sequence.

The maths: if each added nucleotide is a terminator with probability $q$, the fragment length is **geometric**, the same distribution as the stop-codon runs in Lecture 1.1.

**You will:**
1. Simulate chain termination and check the geometric distribution
2. Draw a simulated electropherogram (the coloured trace) and read it
3. See why reads degrade after ~800–1,000 bases
4. Work with **Phred quality scores**
5. Use calculus to find the best ddNTP ratio for a target read length

In [ ]:
import urllib.request, numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(604)

# Template: the first 1,200 bases of phiX174, the first DNA genome ever sequenced (Sanger, 1977)
url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_001422.1&rettype=fasta"
template = "".join(urllib.request.urlopen(url).read().decode().split("\n")[1:])[:1200]
print(template[:80], "...")

## 1. Chain termination is geometric
Each of 100,000 template copies is extended base by base. At every position the polymerase adds a ddNTP (and stops) with probability $q$.

In [ ]:
q = 1 / 300                     # about 1 terminator per 300 nucleotides added
n_molecules = 100_000
lengths = rng.geometric(q, size=n_molecules)       # length at which each chain stopped (1, 2, 3, ...)
lengths = lengths[lengths <= len(template)]

plt.hist(lengths, bins=60, color="slateblue", density=True, label="simulated fragments")
x = np.arange(1, len(template))
plt.plot(x, (1 - q) ** (x - 1) * q, color="red", lw=2, label="geometric: $(1-q)^{L-1}q$")
plt.xlabel("fragment length (bases)"); plt.ylabel("fraction of fragments"); plt.legend(); plt.show()
print(f"mean length ≈ {lengths.mean():.0f}  (theory 1/q = {1/q:.0f}; a little lower here because fragments longer than the 1,200-base template were dropped)")

## 2. The electropherogram
Fragments ending at position $L$ carry the colour of base $L$. The signal at position $L$ is proportional to the number of fragments of that length.
Electrophoresis separates short fragments sharply, but long fragments blur together: we model the peak width growing with length.

In [ ]:
COL = {"A": "green", "C": "blue", "G": "black", "T": "red"}
counts = np.bincount(lengths, minlength=len(template) + 1)

def trace(start, stop):
    grid = np.linspace(start, stop, 2000)
    fig, ax = plt.subplots(figsize=(12, 3))
    signal = {b: np.zeros_like(grid) for b in "ACGT"}
    for L in range(start, stop + 1):
        width = 0.18 + L / 1500          # peaks broaden with fragment length
        signal[template[L - 1]] += counts[L] * np.exp(-0.5 * ((grid - L) / width) ** 2)
    for b in "ACGT":
        ax.plot(grid, signal[b], color=COL[b], lw=1.2, label=b)
    ax.set_xlabel("fragment length = position in sequence"); ax.set_yticks([]); ax.legend(ncol=4)
    ax.set_title(f"positions {start}–{stop}:  true sequence {template[start-1:stop]}")
    plt.show()

trace(20, 45)
trace(700, 725)

✏️ **Exercise 1.** Read the bases from the first trace and compare with the title. Why is the second trace harder to read? Name two separate reasons (signal *and* resolution).

## 3. Phred quality scores
A base caller reports for each base an error probability $p$, written as a **Phred score**
$$Q = -10\log_{10} p \qquad\Longleftrightarrow\qquad p = 10^{-Q/10}$$

In [ ]:
for Q in (10, 20, 30, 40):
    p = 10 ** (-Q / 10)
    print(f"Q{Q}: error probability {p:<7}  →  1 error in {round(1/p):,} bases  (accuracy {100*(1-p):.2f}%)")

✏️ **Exercise 2.** A Sanger read has 750 bases at Q40 and 150 bases at Q15. How many errors do you *expect* in the whole read? (Hint: expected errors = sum of the error probabilities.)

In [ ]:
# your code here

## 4. Choosing the ddNTP ratio (a calculus problem)
The fraction of fragments that stop exactly at position $L$ is $f(q) = (1-q)^{L-1}\,q$.
Too much terminator (large $q$): almost no long fragments. Too little: almost nothing terminates at all.

✏️ **Exercise 3.** Differentiate $f(q)$ with respect to $q$, set it to zero, and show that the best $q$ for reading position $L$ is $q^* = 1/L$. Check it numerically below for $L = 800$.

In [ ]:
L = 800
qs = np.linspace(1e-4, 0.01, 500)
f = (1 - qs) ** (L - 1) * qs
plt.plot(qs, f); plt.axvline(1 / L, color="red", ls="--", label="1/L")
plt.xlabel("q (probability of termination per base)"); plt.ylabel(f"fraction stopping at {L}"); plt.legend(); plt.show()
print("numerical best q:", qs[f.argmax()].round(5), "  theory 1/L:", 1 / L)

---
**Further reading:** Brown, *Genomes 5*, ch. 4 (sequencing genomes) · Pevsner, ch. 9 (next-generation sequence data).